In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from IBL import IBL
from Parser import Parser
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.svm import SVC

from argument_parser import parse_arguments
from csv_writers import create_fw_k_ibl_csv_row, create_k_ibl_csv_row, create_ir_ibl_csv_row, create_svm_csv_row
from model_types import Models
from processing_types import (
    NormalizationStrategy, EncodingStrategy,
    MissingValuesNumericStrategy, MissingValuesCategoricalStrategy, RetentionPolicy
)

BASE_PATH = "../datasetsCBR/datasetsCBR/"
NUM_SPLITS = 1
RESULTS_PATH = "./results/"


if __name__ == "__main__":
    dataset_name = 'adult'

    fw_methods = ['relieff', 'information_gain']

    parser = Parser(
        base_path=BASE_PATH,
        dataset_name=dataset_name,
        normalization_strategy=NormalizationStrategy.MEAN_NORMALIZE,
        encoding_strategy=EncodingStrategy.ONE_HOT_ENCODE, 
        missing_values_numeric_strategy=MissingValuesNumericStrategy.MEDIAN,
        missing_values_categorical_strategy=MissingValuesCategoricalStrategy.MODE,
        num_splits=NUM_SPLITS,
    )
    types = parser.get_types()
    post_encoding_types = parser.get_post_encoding_types()

    splits = [parser.get_split(fold) for fold in range(NUM_SPLITS)]

    all_labels = set()
    for tr, te in splits:
        all_labels.update(np.unique(tr.iloc[:, -1]))
        all_labels.update(np.unique(te.iloc[:, -1]))
    labels = np.array(sorted(all_labels))

    out_csv = Path(RESULTS_PATH +
                   f"{dataset_name}-fw_k_ibl.csv")

    for fw_method in fw_methods:

        rows = []
        for fold_id, (train_matrix, test_matrix) in enumerate(splits):
            # train_matrix = train_matrix.head(1000)
            t0 = time.perf_counter()

        
            ibl = IBL()

            # Fit + predict
            t0 = time.perf_counter()
            
            ibl.fit(train_matrix)
        
            t1 = time.perf_counter()
           
            preds = ibl.fw_KIBLAlgorithm(
                test_matrix,
                k=7,
                metric='cosine',
                vote='borda',
                retention_policy=RetentionPolicy.ALWAYS_RETAIN,
                types=types,
                feature_weighting_method=fw_method,
                post_encoding_types=post_encoding_types[fold_id]
            )

            t2 = time.perf_counter()

            # Times
            fit_time = t1 - t0
            predict_time = t2 - t1
            total_time = t2 - t0

            # Metrics
            y_true = test_matrix.iloc[:, -1].to_numpy()
            y_pred = np.asarray(preds)

            acc = accuracy_score(y_true, y_pred)

            pM, rM, fM, _ = precision_recall_fscore_support(
                y_true, y_pred, average="macro", zero_division=0
            )
            pW, rW, fW, _ = precision_recall_fscore_support(
                y_true, y_pred, average="weighted", zero_division=0
            )

            cm_fold = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

            row = create_fw_k_ibl_csv_row(
                fw_method=fw_method,
                metric='cosine',
                k=7,
                vote='borda',
                retention=RetentionPolicy.ALWAYS_RETAIN,
                fold_id=fold_id,
                num_folds=NUM_SPLITS,
                n_train=train_matrix.shape[0],
                n_test=test_matrix.shape[0],
                fit_time=fit_time,
                predict_time=predict_time,
                total_time=total_time,
                accuracy=acc,
                precision_macro=pM,
                recall_macro=rM,
                f1_macro=fM,
                precision_weighted=pW,
                recall_weighted=rW,
                f1_weighted=fW,
                confusion_matrix=cm_fold,
                labels=labels,
            )
            rows.append(row)

        out_csv.parent.mkdir(parents=True, exist_ok=True)
        df_rows = pd.DataFrame(rows)

        write_header = not out_csv.exists()
        df_rows.to_csv(out_csv, mode="a", header=write_header, index=False)



Instances reduced from 43958 to 43958
Computing feature weights using relieff...
Preallocating matrix of shape (48842, 108)
Total time for all instances: 52.94s

Storage used with specified matrix: 40.24 MB
Final training set size: (48842, 108)
Instances reduced from 43959 to 43959
Computing feature weights using relieff...
Preallocating matrix of shape (48842, 108)
Total time for all instances: 52.95s

Storage used with specified matrix: 40.24 MB
Final training set size: (48842, 108)
Instances reduced from 43957 to 43957
Computing feature weights using relieff...
Preallocating matrix of shape (48842, 108)
Total time for all instances: 52.33s

Storage used with specified matrix: 40.24 MB
Final training set size: (48842, 108)
Instances reduced from 43958 to 43958
Computing feature weights using relieff...
Preallocating matrix of shape (48842, 108)
Total time for all instances: 59.97s

Storage used with specified matrix: 40.24 MB
Final training set size: (48842, 108)
Instances reduced fr